In [10]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [11]:
load_dotenv()

True

In [12]:
model=ChatGroq(model='openai/gpt-oss-120b')

In [13]:
class blogstate(TypedDict):
    title:str
    outline:str
    content:str
    score:float

In [14]:
def create_outline(state:blogstate)->blogstate:
    title=state['title']
    prompt=f"generate a deatailed outline for a blog on the topic {title}"
    outline=model.invoke(prompt).content
    state['outline']=outline
    return state

In [24]:
def create_blog(state:blogstate)->blogstate:
    outline=state['outline']
    title=state['title']
    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'
    content=model.invoke(prompt).content
    state['content']=content
    return state


In [25]:
def score_blog(state:blogstate)->blogstate:
    content=state['content']
    outline=state['outline']
    prompt=f"score the following blog content based on how well it follows the outline and how engaging it is. Provide a score between 1 and 10. \n Outline: {outline} \n Content: {content}"
    score=model.invoke(prompt).content
    state['score']=score
    return state

In [26]:
graph=StateGraph(blogstate)

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('score_blog',score_blog)

graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','score_blog')
graph.add_edge('score_blog',END)

workflow=graph.compile()

In [28]:
intial_state = {'title': 'Rise of AI in India'}
result=workflow.invoke(intial_state)
print(result)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01jxj8qgmnetm8ck1hzk4ak31b` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 8441, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}